In [1]:
import os
import pandas as pd
from transformers import AutoProcessor, AutoModelForVision2Seq
import torch
from PIL import Image
import gc
import re
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# Clear any existing CUDA cache
torch.cuda.empty_cache()
gc.collect()

# Load model in full precision on GPU
print("Loading full model on GPU...")
processor = AutoProcessor.from_pretrained("llava-hf/llava-v1.6-vicuna-7b-hf")
model = AutoModelForVision2Seq.from_pretrained(
    "llava-hf/llava-v1.6-vicuna-7b-hf",
    torch_dtype=torch.float16,
    device_map="cuda"
)

print("Model loaded successfully!")

# Clear cache after loading
torch.cuda.empty_cache()
gc.collect()

# Load CSV with class labels
csv_path = r"D:\Dementia-Thesis - Don't access without permission\2_classes.csv"
df_gt = pd.read_csv(csv_path)

# Normalize ground truth labels to lowercase
if 'dx' in df_gt.columns:
    df_gt['dx'] = df_gt['dx'].apply(lambda x: str(x).lower() if pd.notna(x) else None)

print(f"\nLoaded {len(df_gt)} total records from CSV")

# Filter to only records WITH valid dx labels
df_gt_valid = df_gt[df_gt['dx'].notna()].copy()
print(f"Records with valid dx labels: {len(df_gt_valid)}")
print(f"Records WITHOUT dx labels (will be skipped): {len(df_gt) - len(df_gt_valid)}")
print(f"Unique dx labels: {df_gt_valid['dx'].unique()}")

# Load cookie theft image
image_path = r"D:\Dementia-Thesis - Don't access without permission\Cookie-Theft-stimulus.png"
cookie_image = Image.open(image_path)

# Directory with text files
text_dir = r"D:\Dementia-Thesis - Don't access without permission\cha_par_only"

# Get all text files and filter to only those with valid dx
all_text_files = sorted([f for f in os.listdir(text_dir) if f.endswith('.txt') or f.endswith('.cha')])
print(f"\nFound {len(all_text_files)} total text files")

valid_ids = set(df_gt_valid['id'].astype(str))
text_files = [f for f in all_text_files if f.replace('.txt', '').replace('.cha', '') in valid_ids]

skipped_count = len(all_text_files) - len(text_files)
print(f"Files to process (with valid dx): {len(text_files)}")
print(f"Files skipped (no dx label): {skipped_count}")

# Zero-shot prompt with clear diagnostic criteria
def create_zero_shot_prompt():
    return """Analyze this Cookie Theft picture description for signs of cognitive impairment.

STEP 1: Count how many key elements are mentioned (woman/mother, dishes, sink, water overflowing, children/boy/girl, stool/chair, cookies/jar, reaching/stealing). Record: __/8 elements

STEP 2: Assess linguistic quality:
- Sentence structure: Complete sentences vs fragments?
- Grammar: Correct vs errors/omissions?
- Word-finding: Smooth vs circumlocution ("thing where you wash" instead of "sink")?
- Coherence: Logical narrative vs disconnected phrases?

STEP 3: Determine classification based on THIS patient's evidence:
- CONTROL indicators: 4+ elements identified, complete sentences, good grammar, coherent narrative
- PROBABLEAD indicators: <3 elements identified, fragmented speech, word-finding difficulty, poor coherence

CRITICAL: This dataset has equal numbers of healthy and impaired patients. Do NOT default to one diagnosis. Evaluate objectively.

Respond in this exact format:
Classification: [probablead/control]
MMSE: [0-30]"""

results = []

print("\n" + "="*80)
print("ZERO-SHOT CLASSIFICATION - PROCESSING FILES")
print("="*80)

processed_count = 0

for idx, filename in enumerate(text_files):
    file_id = filename.replace('.txt', '').replace('.cha', '')
    
    print(f"\n{'='*80}")
    print(f"Processing: {filename} (ID: {file_id})")
    print('='*80)
    
    try:
        # Read text file
        file_path = os.path.join(text_dir, filename)
        with open(file_path, 'r', encoding='utf-8') as f:
            text_content = f.read().strip()
        
        # Check for empty/short transcription
        if len(text_content.strip()) < 10:
            print(f"⚠️ Empty/short transcription - marking as 'error'")
            results.append({
                'id': file_id,
                'filename': filename,
                'predicted_diagnosis': 'error',
                'predicted_mmse': -1,
                'raw_output': 'Transcription too short or empty',
                'text_snippet': text_content[:150]
            })
            processed_count += 1
            continue
        
        print(f"Transcription preview: {text_content[:200]}...")
        
        # Create messages
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": f"Description: {text_content}\n\n{create_zero_shot_prompt()}"}
                ]
            }
        ]
        
        # Process inputs
        prompt_text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        inputs = processor(
            text=prompt_text,
            images=cookie_image,
            return_tensors="pt"
        ).to(model.device)
        
        # Generate prediction with variability
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=True,
                temperature=1.0,
                top_p=0.92,
                top_k=50
            )
        
        # Decode output
        generated_text = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
        
        # Clear cache after generation
        del inputs, outputs
        torch.cuda.empty_cache()
        
        # Extract classification and MMSE prediction
        prediction = None
        predicted_mmse = -1
        
        # Strategy 1: Look for "Classification:" followed by the label
        class_match = re.search(r'Classification:\s*(\bprobablead\b|\bcontrol\b)', generated_text, re.IGNORECASE)
        if class_match:
            prediction = class_match.group(1).lower()
        
        # Strategy 2: Look anywhere in first 200 chars
        if not prediction:
            early_match = re.search(r'(\bprobablead\b|\bcontrol\b)', generated_text[:200], re.IGNORECASE)
            if early_match:
                prediction = early_match.group(1).lower()
        
        # Strategy 3: Look for variations
        if not prediction:
            if re.search(r'\balzheimer\b|\bad\b', generated_text, re.IGNORECASE):
                prediction = 'probablead'
            elif re.search(r'\bhealthy\b|\bnormal\b|\bcontrol\b', generated_text, re.IGNORECASE):
                prediction = 'control'
        
        # MMSE extraction
        mmse_match = re.search(r'MMSE:\s*(\d+)', generated_text, re.IGNORECASE)
        if mmse_match:
            try:
                candidate_score = int(mmse_match.group(1))
                if 0 <= candidate_score <= 30:
                    predicted_mmse = candidate_score
            except ValueError:
                pass
        
        # If still no MMSE, try broader search
        if predicted_mmse == -1:
            mmse_pattern = re.search(r'(?:score|mmse|estimate).*?(\d+)', generated_text, re.IGNORECASE)
            if mmse_pattern:
                try:
                    candidate_score = int(mmse_pattern.group(1))
                    if 0 <= candidate_score <= 30:
                        predicted_mmse = candidate_score
                except ValueError:
                    pass
        
        # Final check - if still no prediction, mark as 'unknown'
        if not prediction:
            prediction = 'unknown'
            print(f"⚠️ No diagnosis found in model output, marked as 'unknown'")
            print(f"   Model output was: {generated_text[:300]}...")
        
        results.append({
            'id': file_id,
            'filename': filename,
            'predicted_diagnosis': prediction,
            'predicted_mmse': predicted_mmse,
            'raw_output': generated_text,
            'text_snippet': text_content[:150]
        })
        
        processed_count += 1
        print(f"✅ Processed: {prediction} (MMSE: {predicted_mmse if predicted_mmse != -1 else 'unknown'})")
        
        # Periodic garbage collection
        if processed_count % 10 == 0:
            gc.collect()
            torch.cuda.empty_cache()
            
    except Exception as e:
        print(f"❌ ERROR processing {filename}: {str(e)}")
        print(f"Marking as 'error' with MMSE=-1")
        import traceback
        traceback.print_exc()
        
        results.append({
            'id': file_id,
            'filename': filename,
            'predicted_diagnosis': 'error',
            'predicted_mmse': -1,
            'raw_output': f'Processing error: {str(e)[:100]}',
            'text_snippet': ''
        })
        processed_count += 1
        continue

print(f"\n{'='*80}")
print(f"PROCESSING COMPLETE!")
print(f"Total files processed: {len(results)}")
print('='*80)

# Create DataFrame
df_results = pd.DataFrame(results)

# Count error/unknown predictions
error_count = (df_results['predicted_diagnosis'] == 'error').sum()
unknown_count = (df_results['predicted_diagnosis'] == 'unknown').sum()
valid_count = len(df_results) - error_count - unknown_count

print(f"\nResults breakdown:")
print(f"  Valid predictions: {valid_count}")
print(f"  Unknown predictions: {unknown_count}")
print(f"  Error predictions: {error_count}")
print(f"  Total: {len(df_results)}")

# ============================================================================
# MERGE WITH GROUND TRUTH AND CALCULATE METRICS
# ============================================================================
print(f"\n{'='*80}")
print("CALCULATING METRICS")
print('='*80)

# Merge predictions with ground truth
df_merged = df_results.merge(df_gt_valid, on='id', how='inner')
print(f"\nMerged {len(df_merged)} records with ground truth")

if len(df_merged) > 0:
    # For accuracy calculation, error/unknown count as wrong
    y_true = df_merged['dx'].values
    y_pred = df_merged['predicted_diagnosis'].values
    
    print(f"\nUnique true labels: {np.unique(y_true)}")
    print(f"Unique predicted labels: {np.unique(y_pred)}")
    
    # Count predictions
    correct_predictions = (y_true == y_pred).sum()
    total_predictions = len(df_merged)
    accuracy_all = correct_predictions / total_predictions
    
    error_unknown_count = df_merged['predicted_diagnosis'].isin(['error', 'unknown']).sum()
    
    print(f"\n--- CLASSIFICATION METRICS (error/unknown count as WRONG) ---")
    print(f"Total with ground truth: {total_predictions}")
    print(f"Correct predictions: {correct_predictions}")
    print(f"Wrong predictions: {total_predictions - correct_predictions}")
    print(f"  - of which error/unknown: {error_unknown_count}")
    print(f"Accuracy (ALL): {accuracy_all:.4f}")
    
    # Calculate standard metrics only on valid predictions
    df_valid = df_merged[~df_merged['predicted_diagnosis'].isin(['error', 'unknown'])].copy()
    
    if len(df_valid) > 0:
        y_true_valid = df_valid['dx'].values
        y_pred_valid = df_valid['predicted_diagnosis'].values
        
        accuracy_valid = accuracy_score(y_true_valid, y_pred_valid)
        precision, recall, f1, support = precision_recall_fscore_support(
            y_true_valid, y_pred_valid,
            average='binary',
            pos_label='probablead',
            zero_division=0
        )
        
        # Confusion Matrix
        cm = confusion_matrix(y_true_valid, y_pred_valid, labels=['probablead', 'control'])
        tp, fn, fp, tn = cm.ravel()
        
        # Specificity and Sensitivity
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        print(f"\n--- METRICS ON VALID PREDICTIONS ONLY (excluding error/unknown) ---")
        print(f"Valid predictions evaluated: {len(df_valid)}")
        print(f"Accuracy:    {accuracy_valid:.4f}")
        print(f"Precision:   {precision:.4f}")
        print(f"Recall:      {recall:.4f}")
        print(f"F1-Score:    {f1:.4f}")
        print(f"Sensitivity: {sensitivity:.4f}")
        print(f"Specificity: {specificity:.4f}")
        
        print("\nConfusion Matrix (valid predictions only):")
        print(f"                  Predicted")
        print(f"Actual       probableAD  control")
        print(f"probableAD     {tp:5d}    {fn:5d}")
        print(f"control        {fp:5d}    {tn:5d}")
    else:
        print("\n⚠️ No valid predictions to calculate detailed metrics")
        accuracy_valid = 0
        precision = recall = f1 = sensitivity = specificity = 0
        tp = tn = fp = fn = 0
    
    # MMSE Metrics
    records_with_mmse = df_merged[df_merged['mmse'].notna()].copy()
    
    if len(records_with_mmse) > 0:
        correct_mmse = (records_with_mmse['mmse'] == records_with_mmse['predicted_mmse']).sum()
        total_mmse = len(records_with_mmse)
        error_mmse = (records_with_mmse['predicted_mmse'] == -1).sum()
        
        print(f"\n--- MMSE METRICS (error/-1 counts as WRONG) ---")
        print(f"Total with ground truth MMSE: {total_mmse}")
        print(f"Exact matches: {correct_mmse}")
        print(f"Wrong predictions: {total_mmse - correct_mmse}")
        print(f"  - of which error/unknown (-1): {error_mmse}")
        
        # For MAE/RMSE, only use valid predictions
        df_valid_mmse = records_with_mmse[records_with_mmse['predicted_mmse'] >= 0]
        
        if len(df_valid_mmse) > 0:
            from sklearn.metrics import mean_absolute_error, mean_squared_error
            
            y_true_mmse = df_valid_mmse['mmse'].values
            y_pred_mmse = df_valid_mmse['predicted_mmse'].values
            
            mae = mean_absolute_error(y_true_mmse, y_pred_mmse)
            mse = mean_squared_error(y_true_mmse, y_pred_mmse)
            rmse = np.sqrt(mse)
            
            print(f"\nRegression metrics (valid predictions only, excluding -1):")
            print(f"MAE (Mean Absolute Error): {mae:.4f}")
            print(f"MSE (Mean Squared Error):  {mse:.4f}")
            print(f"RMSE (Root MSE):           {rmse:.4f}")
            print(f"Samples evaluated:         {len(df_valid_mmse)}")
        else:
            print("\nNo valid MMSE predictions (all were -1/error)")
            mae = mse = rmse = np.nan
    else:
        print("\n--- MMSE METRICS ---")
        print("No records with ground truth MMSE")
        mae = mse = rmse = np.nan
        correct_mmse = total_mmse = error_mmse = 0
    
    # Create metrics summary rows
    metrics_summary = [
        {'id': '', 'filename': '--- METRICS SUMMARY ---', 'predicted_diagnosis': '', 'predicted_mmse': np.nan, 'raw_output': '', 'text_snippet': ''},
        {'id': 'METRIC', 'filename': 'Accuracy (ALL, error=wrong)', 'predicted_diagnosis': '', 'predicted_mmse': accuracy_all, 'raw_output': '', 'text_snippet': ''},
        {'id': 'METRIC', 'filename': 'Accuracy (valid only)', 'predicted_diagnosis': '', 'predicted_mmse': accuracy_valid, 'raw_output': '', 'text_snippet': ''},
        {'id': 'METRIC', 'filename': 'Precision', 'predicted_diagnosis': '', 'predicted_mmse': precision, 'raw_output': '', 'text_snippet': ''},
        {'id': 'METRIC', 'filename': 'Recall', 'predicted_diagnosis': '', 'predicted_mmse': recall, 'raw_output': '', 'text_snippet': ''},
        {'id': 'METRIC', 'filename': 'F1-Score', 'predicted_diagnosis': '', 'predicted_mmse': f1, 'raw_output': '', 'text_snippet': ''},
        {'id': 'METRIC', 'filename': 'Sensitivity', 'predicted_diagnosis': '', 'predicted_mmse': sensitivity, 'raw_output': '', 'text_snippet': ''},
        {'id': 'METRIC', 'filename': 'Specificity', 'predicted_diagnosis': '', 'predicted_mmse': specificity, 'raw_output': '', 'text_snippet': ''},
        {'id': '', 'filename': '', 'predicted_diagnosis': '', 'predicted_mmse': np.nan, 'raw_output': '', 'text_snippet': ''},
        {'id': 'CONFUSION', 'filename': 'True Positives', 'predicted_diagnosis': '', 'predicted_mmse': int(tp), 'raw_output': '', 'text_snippet': ''},
        {'id': 'CONFUSION', 'filename': 'True Negatives', 'predicted_diagnosis': '', 'predicted_mmse': int(tn), 'raw_output': '', 'text_snippet': ''},
        {'id': 'CONFUSION', 'filename': 'False Positives', 'predicted_diagnosis': '', 'predicted_mmse': int(fp), 'raw_output': '', 'text_snippet': ''},
        {'id': 'CONFUSION', 'filename': 'False Negatives', 'predicted_diagnosis': '', 'predicted_mmse': int(fn), 'raw_output': '', 'text_snippet': ''},
        {'id': '', 'filename': '', 'predicted_diagnosis': '', 'predicted_mmse': np.nan, 'raw_output': '', 'text_snippet': ''},
        {'id': 'MMSE', 'filename': 'MAE', 'predicted_diagnosis': '', 'predicted_mmse': mae, 'raw_output': '', 'text_snippet': ''},
        {'id': 'MMSE', 'filename': 'MSE', 'predicted_diagnosis': '', 'predicted_mmse': mse, 'raw_output': '', 'text_snippet': ''},
        {'id': 'MMSE', 'filename': 'RMSE', 'predicted_diagnosis': '', 'predicted_mmse': rmse, 'raw_output': '', 'text_snippet': ''},
        {'id': '', 'filename': '', 'predicted_diagnosis': '', 'predicted_mmse': np.nan, 'raw_output': '', 'text_snippet': ''},
        {'id': 'COUNT', 'filename': 'Total Predictions', 'predicted_diagnosis': '', 'predicted_mmse': len(df_results), 'raw_output': '', 'text_snippet': ''},
        {'id': 'COUNT', 'filename': 'Valid DX Predictions', 'predicted_diagnosis': '', 'predicted_mmse': len(df_valid) if len(df_valid) > 0 else 0, 'raw_output': '', 'text_snippet': ''},
        {'id': 'COUNT', 'filename': 'Error/Unknown DX', 'predicted_diagnosis': '', 'predicted_mmse': error_unknown_count, 'raw_output': '', 'text_snippet': ''},
        {'id': 'COUNT', 'filename': 'Valid MMSE Predictions', 'predicted_diagnosis': '', 'predicted_mmse': len(df_valid_mmse) if len(records_with_mmse) > 0 and len(df_valid_mmse) > 0 else 0, 'raw_output': '', 'text_snippet': ''},
        {'id': 'COUNT', 'filename': 'Error/Unknown MMSE (-1)', 'predicted_diagnosis': '', 'predicted_mmse': error_mmse, 'raw_output': '', 'text_snippet': ''},
    ]
    
    df_with_metrics = pd.concat([df_results, pd.DataFrame(metrics_summary)], ignore_index=True)
    
else:
    print("\n⚠️ No records with ground truth to evaluate!")
    df_with_metrics = df_results

# Save results
output_path = r"D:\Dementia-Thesis - Don't access without permission\llava_prompting.csv"
try:
    df_with_metrics.to_csv(output_path, index=False)
    print(f"\n{'='*80}")
    print(f"✅ PREDICTIONS AND METRICS SAVED TO CSV!")
    print(f"Output file: {output_path}")
    print(f"Total records saved: {len(df_with_metrics)}")
    print(f"(Includes {len(df_results)} predictions + metrics summary)")
    print(f"\nCSV includes:")
    print(f"  - Valid predictions: {valid_count}")
    print(f"  - Unknown predictions (pred_dx='unknown', pred_mmse=-1): {unknown_count}")
    print(f"  - Error predictions (pred_dx='error', pred_mmse=-1): {error_count}")
    print(f"\nNote: {skipped_count} files were skipped (no dx label in ground truth)")
    print('='*80)
except Exception as e:
    print(f"\n❌ ERROR saving CSV: {str(e)}")
    print("Results are still available in the df_results DataFrame")

print("\n" + "="*80)
print("Script execution completed!")
print("="*80)

c:\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading full model on GPU...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
c:\Python314\Lib\site-packages\transformers\models\auto\modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:12<00:00,  4.08s/it]


Model loaded successfully!

Loaded 498 total records from CSV
Records with valid dx labels: 498
Records WITHOUT dx labels (will be skipped): 0
Unique dx labels: ['probablead' 'control']

Found 552 total text files
Files to process (with valid dx): 498
Files skipped (no dx label): 54

ZERO-SHOT CLASSIFICATION - PROCESSING FILES

Processing: 001-0.cha (ID: 001-0)
Transcription preview: mhm . [+ exc] 
+< alright . [+ exc] 
there's &-um a young boy that's getting a cookie jar . 
and it [//] he's &-uh in bad shape because &-uh the thing is fallin(g) over . 
and in the picture the mothe...
✅ Processed: probablead (MMSE: 30)

Processing: 001-2.cha (ID: 001-2)
Transcription preview: mhm . 
there's a young boy &-uh going in a cookie jar . 
and there's a [/] &+lit a girl [//] young girl . 
and I'm sayin(g) he's a boy (be)cause <you can> [//] &+hard it's hardly [//] hard to tell any...
✅ Processed: probablead (MMSE: 25)

Processing: 002-0.cha (ID: 002-0)
Transcription preview: the scene is <in th